# Model A — ResNet-50 CNN classifier
**Owner:** Lakshya Jain

Transfer-learning ResNet-50 classifies each ground-truth damage crop into D00/D10/D20/D40. ~35–50 min on a T4.

**Before running:** Kaggle → *Settings* → Accelerator **GPU T4 x2**, Internet **On**. *Add Input* → dataset **aliabdelmenam/rdd-2022** (the same RDD2022 `RDD_SPLIT` used in Phase 2). Then *Run All*.
The final cell zips `results/` → download it from the Output panel and put it into the repo's `results/` folder.

In [ ]:
import os, subprocess, sys
# Option A (recommended): the final code is pushed to the team repo
if not os.path.exists('repo'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/lakshya1333/Road-damage-detection-deep-learning-project', 'repo'], check=True)
# Option B: if you uploaded the code folder as a Kaggle dataset instead, copy it:  !cp -r /kaggle/input/<your-code-dataset> repo
os.chdir('repo')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics>=8.2', 'segmentation-models-pytorch>=0.3'], check=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Shared manifest: identical split + class mapping for all four models (check the printed class mapping!)
!python -m src.data.build_manifest --out results/data --plot

In [ ]:
!python -m src.models.train_classifier --arch resnet50 --manifest results/data --out results/resnet50 --epochs 12 --batch 64

In [ ]:
# optional lightweight variant for the edge-deployment discussion (~20 min)
# !python -m src.models.train_classifier --arch mobilenet_v2 --manifest results/data --out results/mobilenet_v2 --epochs 12 --batch 128

In [ ]:
from IPython.display import Image, display
display(Image('results/resnet50/training_curves.png'))
display(Image('results/resnet50/confusion_matrix.png'))

In [ ]:
import shutil, glob, os
# drop large weights from the archive except the best checkpoint
shutil.make_archive('/kaggle/working/results_resnet50', 'zip', 'results')
print(open(sorted(glob.glob('results/*/metrics.json'))[0]).read()[:3000])